In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install notebook dependencies in Colab runtime.
    %pip -q install rank-bm25

    from google.colab import drive
    drive.mount("/content/drive")
        

# Optional override if your repo root is non-standard.
# Example: /content/drive/MyDrive/Legal-question-answer-v1
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "")
if PROJECT_ROOT:
    print(f"Using PROJECT_ROOT from environment: {PROJECT_ROOT}")

In [2]:
import sys
from pathlib import Path

# Resolve project root in both local and Colab environments.
candidate_roots = []

# 1) Explicit env override from setup cell
project_root_env = Path(PROJECT_ROOT).expanduser() if "PROJECT_ROOT" in globals() and PROJECT_ROOT else None
if project_root_env:
    candidate_roots.append(project_root_env)

# 2) Current working directory search upward
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

# 3) Common Colab locations
candidate_roots.extend([
    Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
])

project_root = None
for path in candidate_roots:
    if (path / "utils").exists() and (path / "data").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find project root containing 'utils/' and 'data/'. "
        "Set PROJECT_ROOT env var, or update the Colab setup cell path."
    )

utils_path = str(project_root / "utils")
if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Resolved project_root: {project_root}")

from get_jsonl_data import get_jsonl_data

Resolved project_root: E:\AI\NLP\Legal-question-answer-v1


In [3]:
corpus_path = project_root / "data" / "corpus.jsonl"
chunk_store_path = project_root / "data" / "chunk_store.jsonl"

corpus = get_jsonl_data(str(corpus_path))
chunk_store = get_jsonl_data(str(chunk_store_path))

print(corpus[0])
print(chunk_store[0])

{'doc_id': 'doc_0', 'article_id': 'article_0', 'article_content': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.\n\n2. Tranh chấp quốc tế về địa chất, khoáng sản được giải quyết thông qua các biện pháp hòa bình, theo thông lệ quốc tế, pháp luật quốc tế và pháp luật của các bên liên quan.

In [4]:
from collections import defaultdict
from rank_bm25 import BM25Okapi
import numpy as np

def normalize_whitespace(text):
    return " ".join((text or "").split())

top_k = 100
top_k_pos = 5
progress_every_rows = 100

# Pre-index chunks by article to avoid repeated linear scans.
chunks_by_article = defaultdict(list)
for chunk in chunk_store:
    article_id = chunk.get("article_id")
    chunk_text = normalize_whitespace(chunk.get("text"))
    if article_id is not None and chunk_text:
        chunks_by_article[article_id].append(chunk_text)

# Build BM25 once per article and cache it.
bm25_cache = {}
for article_id, chunk_texts in chunks_by_article.items():
    tokenized_chunks = [chunk.split() for chunk in chunk_texts]
    if tokenized_chunks:
        bm25_cache[article_id] = (chunk_texts, BM25Okapi(tokenized_chunks))

training_data = []

total_rows = len(corpus)
rows_with_cache = 0
qa_seen = 0
qa_valid = 0

print(f"Starting positive chunk generation for {total_rows} corpus rows...")

for row_idx, row in enumerate(corpus, start=1):
    doc_id = row.get("doc_id")
    article_id = row.get("article_id")
    qa_pairs = row.get("generated_qa_pairs", [])

    # Skip rows with no valid chunks for the article.
    if article_id not in bm25_cache:
        if row_idx % progress_every_rows == 0 or row_idx == total_rows:
            print(
                f"[Progress] rows={row_idx}/{total_rows} | rows_with_cache={rows_with_cache} | "
                f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
            )
        continue

    rows_with_cache += 1
    chunk_texts, bm25 = bm25_cache[article_id]

    for qa_pair in qa_pairs:
        qa_seen += 1

        question = normalize_whitespace(qa_pair.get("question"))
        answer = normalize_whitespace(qa_pair.get("answer"))

        # Skip malformed QA entries.
        if not question or not answer:
            continue

        qa_valid += 1

        tokenized_question = question.split()
        bm25_scores = bm25.get_scores(tokenized_question)
        if len(bm25_scores) == 0:
            continue

        k = min(top_k, len(bm25_scores))
        if k == len(bm25_scores):
            top_k_idx = np.argsort(bm25_scores)[::-1]
        else:
            top_k_idx = np.argpartition(bm25_scores, -k)[-k:]
            top_k_idx = top_k_idx[np.argsort(bm25_scores[top_k_idx])[::-1]]

        top_k_chunks = [chunk_texts[idx] for idx in top_k_idx]
        top_k_positive_chunks = top_k_chunks[:top_k_pos]

        sample = {
            "doc_id": doc_id,
            "article_id": article_id,
            "question": question,
            "answer": answer,
            "positive_chunks": top_k_positive_chunks,
        }
        training_data.append(sample)

    if row_idx % progress_every_rows == 0 or row_idx == total_rows:
        print(
            f"[Progress] rows={row_idx}/{total_rows} | rows_with_cache={rows_with_cache} | "
            f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
        )

print(
    f"Done. rows={total_rows}, rows_with_cache={rows_with_cache}, "
    f"qa_seen={qa_seen}, qa_valid={qa_valid}, samples={len(training_data)}"
)

Starting positive chunk generation for 9715 corpus rows...
[Progress] rows=100/9715 | rows_with_cache=100 | qa_seen=300 | qa_valid=300 | samples=300
[Progress] rows=200/9715 | rows_with_cache=200 | qa_seen=600 | qa_valid=600 | samples=600
[Progress] rows=300/9715 | rows_with_cache=300 | qa_seen=900 | qa_valid=900 | samples=900
[Progress] rows=400/9715 | rows_with_cache=400 | qa_seen=1200 | qa_valid=1200 | samples=1200
[Progress] rows=500/9715 | rows_with_cache=500 | qa_seen=1500 | qa_valid=1500 | samples=1500
[Progress] rows=600/9715 | rows_with_cache=600 | qa_seen=1800 | qa_valid=1800 | samples=1800
[Progress] rows=700/9715 | rows_with_cache=700 | qa_seen=2100 | qa_valid=2100 | samples=2100
[Progress] rows=800/9715 | rows_with_cache=800 | qa_seen=2400 | qa_valid=2400 | samples=2400
[Progress] rows=900/9715 | rows_with_cache=900 | qa_seen=2700 | qa_valid=2700 | samples=2700
[Progress] rows=1000/9715 | rows_with_cache=1000 | qa_seen=3000 | qa_valid=3000 | samples=3000
[Progress] rows=11

In [5]:
print("Number of training samples:", len(training_data))
if training_data:
    print(training_data[0])
else:
    print("No training samples were created. Check corpus/chunk quality and filtering conditions.")

Number of training samples: 29145
{'doc_id': 'doc_0', 'article_id': 'article_0', 'question': 'Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?', 'answer': 'Theo Khoản 1 Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong các hoạt động sau: nghiên cứu, điều tra cơ bản địa chất; điều tra địa chất về khoáng sản; hoạt động khoáng sản; và quản lý hoạt động khoáng sản.', 'positive_chunks': ['1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng;', 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản', '2. Tranh chấp quốc tế về địa chất, khoáng sản được giải quyết thông q

In [6]:
# Save the training data to a JSONL file
import json

output_path = project_root / "data" / "training_data.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")

Saved to: E:\AI\NLP\Legal-question-answer-v1\data\training_data.jsonl
